In [27]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
import os
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
import requests
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing import Annotated
import random


load_dotenv()  # Load environment variables from .env file

True

In [28]:
model = ChatGroq(
    model_name="openai/gpt-oss-120b",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0.7,
    streaming=True,
    stop=None,
)

In [29]:
class ChatState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages]

In [30]:
#tools 
#prebuild tools 
search_tool =DuckDuckGoSearchRun(region="us-en")

#coustom tools
@tool
def calculator(first: float, second: float, operation: str) -> int | float:
    """Perform a basic arithmetic calculation on two integers.

    Supported operations:
    - add
    - subtract
    - multiply
    - divide
    - modulo
    - power
    """
    op = operation.strip().lower()

    if op == "add":
        return first + second
    if op == "subtract":
        return first - second
    if op == "multiply":
        return first * second
    if op == "divide":
        if second == 0:
            raise ValueError("Division by zero is not allowed.")
        return first / second
    if op == "modulo":
        if second == 0:
            raise ValueError("Modulo by zero is not allowed.")
        return first % second
    if op == "power":
        return first ** second

    raise ValueError(
        "Unsupported operation. Use one of: add, subtract, multiply, divide, modulo, power."
    )

#coustom tools
@tool
def get_stock_prices(company: str) -> dict:
    """Get the latest stock price for a company name or ticker symbol using Alpha Vantage."""
    api_key = os.getenv("ALPHAVANTAGE_API_KEY")
    if not api_key:
        raise ValueError("Set the ALPHAVANTAGE_API_KEY environment variable.")

    def fetch_quote(symbol: str) -> dict:
        response = requests.get(
            "https://www.alphavantage.co/query",
            params={
                "function": "GLOBAL_QUOTE",
                "symbol": symbol,
                "apikey": api_key,
            },
            timeout=15,
        )
        response.raise_for_status()
        data = response.json()

        if "Note" in data or "Information" in data:
            raise RuntimeError(data.get("Note") or data["Information"])

        quote = data.get("Global Quote")
        if not quote:
            raise ValueError(f"No stock data found for '{symbol}'.")

        return {
            "symbol": quote.get("01. symbol"),
            "price": float(quote["05. price"]),
            "change": quote.get("09. change"),
            "change_percent": quote.get("10. change percent"),
            "latest_trading_day": quote.get("07. latest trading day"),
        }

    # Try the provided value as a ticker first.
    try:
        return fetch_quote(company.strip().upper())
    except ValueError:
        pass

    # Otherwise search for the company name.
    response = requests.get(
        "https://www.alphavantage.co/query",
        params={
            "function": "SYMBOL_SEARCH",
            "keywords": company,
            "apikey": api_key,
        },
        timeout=15,
    )
    response.raise_for_status()
    matches = response.json().get("bestMatches", [])

    if not matches:
        raise ValueError(f"Company '{company}' was not found.")

    symbol = matches[0]["1. symbol"]
    return fetch_quote(symbol)


In [31]:
# create the list of tools
tools = [search_tool, calculator, get_stock_prices]
llm_with_tools=model.bind_tools(tools)

# pass them to ToolNode
tool_node = ToolNode(tools)

In [32]:
graph = StateGraph(ChatState)

In [33]:
def ChatGroq_llm(state: ChatState):
    system_message = SystemMessage(
        content="You are a helpful assistant. Use tools when they can improve your answer."
    )
    bot_response = llm_with_tools.invoke([system_message, *state.messages])
    return {"messages": [bot_response]}

In [34]:

graph.add_node("LLm_node", ChatGroq_llm)
graph.add_node("tools",tool_node)


In [36]:
graph.add_edge(START, "LLm_node")
graph.add_conditional_edges("LLm_node", tools_condition)
graph.add_edge("tools", "LLm_node")
chatbot = graph.compile()

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


ValueError: Branch with name `tools_condition` already exists for node `LLm_node`